In [ ]:
"""
DIAGNÓSTICO DE TimeSeriesSplit — Solo consola, sin guardar nada
================================================================
Permite observar cómo el TimeSeriesSplit divide los datos parasitarios
en bloques de entrenamiento y prueba respetando el orden temporal.

Muestra:
  1. Rango de fechas y conteo de registros con parásitos
  2. Distribución por centro y año
  3. Cómo queda cada fold (train/test, fechas, tamaños)
  4. Composición por centro dentro de cada fold de test

NOTA: Las variables temporales (Mes, Semana, etc.) NO se usan como
features del modelo. La fecha solo se emplea para ordenar y dividir.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# ============================================================
# CONFIGURACIÓN — cambia aquí para experimentar
# ============================================================
RUTA = "data/Dataset_Maestro_Amplificado_ML.csv"   # ajusta la ruta a tu archivo
TARGET = "Hembras_Ovigeras"   # o "Total_Parasitos_"
N_SPLITS = 5                  # número de folds del TimeSeriesSplit
# ============================================================

# ---------- 1. Carga y filtrado ----------
df = pd.read_csv(RUTA)
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Solo registros con parásitos (los únicos válidos para entrenar)
cp = df[df[TARGET].notna()].copy()

# CLAVE: ordenar cronológicamente antes de cualquier split temporal
cp = cp.sort_values('Fecha').reset_index(drop=True)

print("=" * 75)
print("1. RESUMEN GENERAL")
print("=" * 75)
print(f"Registros totales en el dataset : {len(df)}")
print(f"Registros con parásitos ({TARGET}): {len(cp)}")
print(f"Rango de fechas con parásitos    : {cp['Fecha'].min().date()} → {cp['Fecha'].max().date()}")
print(f"Última fecha oceanográfica       : {df['Fecha'].max().date()}")

# ---------- 2. Distribución por centro y año ----------
cp['Año'] = cp['Fecha'].dt.year
print("\n" + "=" * 75)
print("2. REGISTROS CON PARÁSITOS POR CENTRO Y AÑO")
print("=" * 75)
tabla = pd.crosstab(cp['id_centro'], cp['Año'], margins=True, margins_name='TOTAL')
print(tabla.to_string())

print("\nRegión de cada centro:")
print(cp.groupby('id_centro')['Region'].first().to_string())

# ---------- 3. Los folds del TimeSeriesSplit ----------
print("\n" + "=" * 75)
print(f"3. TimeSeriesSplit CON {N_SPLITS} FOLDS")
print("=" * 75)
print(f"{'Fold':<6}{'Train n':<10}{'Test n':<9}{'Train fechas':<28}{'Test fechas'}")
print("-" * 75)

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
folds = list(tscv.split(cp))

for i, (train_idx, test_idx) in enumerate(folds):
    tr = cp.iloc[train_idx]
    te = cp.iloc[test_idx]
    tr_range = f"{tr['Fecha'].min().date()} a {tr['Fecha'].max().date()}"
    te_range = f"{te['Fecha'].min().date()} a {te['Fecha'].max().date()}"
    print(f"{i+1:<6}{len(tr):<10}{len(te):<9}{tr_range:<28}{te_range}")

print("\nNota: el train siempre empieza en la fecha más antigua y crece;")
print("el test es siempre el período inmediatamente posterior (predice hacia adelante).")

# ---------- 4. Composición por centro en cada test ----------
print("\n" + "=" * 75)
print("4. COMPOSICIÓN POR CENTRO EN CADA TEST FOLD")
print("=" * 75)
for i, (train_idx, test_idx) in enumerate(folds):
    te = cp.iloc[test_idx]
    comp = te['id_centro'].value_counts().sort_index()
    comp_str = "   ".join([f"{c}: {n} ({100*n/len(te):.0f}%)" for c, n in comp.items()])
    print(f"Fold {i+1} (n={len(te)}): {comp_str}")

# ---------- 5. Distribución del target en cada test ----------
print("\n" + "=" * 75)
print(f"5. ESTADÍSTICAS DEL TARGET ({TARGET}) EN CADA TEST FOLD")
print("=" * 75)
print(f"{'Fold':<6}{'media':<9}{'mediana':<9}{'min':<7}{'max':<8}{'% ceros':<9}")
print("-" * 50)
for i, (train_idx, test_idx) in enumerate(folds):
    te = cp.iloc[test_idx]
    y = te[TARGET]
    pct_ceros = 100 * (y == 0).sum() / len(y)
    print(f"{i+1:<6}{y.mean():<9.3f}{y.median():<9.3f}{y.min():<7.2f}{y.max():<8.2f}{pct_ceros:<9.1f}")

print("\n" + "=" * 75)
print("Observa: si un fold tiene composición muy desbalanceada o un % de ceros")
print("muy distinto, sus métricas pueden diferir del resto. Eso es información,")
print("no un error — refleja la estructura temporal real de tus datos.")
print("=" * 75)

In [ ]:
"""
==============================================================================
 PIPELINE BASELINE — RANDOM FOREST (Regresión)
==============================================================================
 Objetivo: establecer un modelo de referencia con hiperparámetros por
 defecto para predecir la prevalencia parasitaria a partir de variables
 físicas oceanográficas.

 Diseño modular por bloques independientes:
   BLOQUE 1 — Configuración (única zona a editar)
   BLOQUE 2 — Carga y preparación de datos
   BLOQUE 3 — Estrategia de validación (temporal / aleatoria)
   BLOQUE 4 — Entrenamiento y evaluación
   BLOQUE 5 — Reporte por consola
   BLOQUE 6 — Orquestador (main)

 Sin optimización de hiperparámetros. Sin guardar archivos.
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN  (lo único que editas para experimentar)
# ============================================================================
CONFIG = {
    # --- Datos ---
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Hembras_Ovigeras",       #Hembras_Ovigeras o "Total_Parasitos_"
    "features"   : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
        "Adultos_Moviles",
        "Juveniles"
        
    ],

    # --- Validación ---
    "validacion" : "aleatoria",   # "temporal" (TimeSeriesSplit) o "aleatoria" (KFold)
    "n_folds"    : 5,
    "random_state": 42,

    # --- Hiperparámetros del modelo ---
    # Vacío {} = valores por defecto de scikit-learn (baseline limpio)
    "rf_params"  : {
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN DE DATOS
# ============================================================================
def cargar_datos(config):
    """
    Lee el CSV, filtra registros con target presente (descarta descansos),
    ordena cronológicamente y devuelve X, y y el dataframe de referencia.
    """
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    # Solo registros con el target medido (los únicos que sirven para entrenar)
    df = df[df[config["target"]].notna()].copy()

    # Orden cronológico: imprescindible para validación temporal
    df = df.sort_values("Fecha").reset_index(drop=True)

    X = df[config["features"]].copy()
    y = df[config["target"]].copy()

    return X, y, df


# ============================================================================
# BLOQUE 3 — ESTRATEGIA DE VALIDACIÓN
# ============================================================================
def obtener_folds(config, X):
    """
    Devuelve un iterable de (train_idx, test_idx) según la estrategia elegida.
      - "temporal"  : TimeSeriesSplit (respeta el orden del tiempo)
      - "aleatoria" : KFold con shuffle (observaciones independientes)
    """
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
    elif config["validacion"] == "aleatoria":
        splitter = KFold(
            n_splits=config["n_folds"],
            shuffle=True,
            random_state=config["random_state"],
        )
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")

    return list(splitter.split(X))


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    """
    Entrena un RandomForestRegressor en cada fold y calcula métricas.
    Devuelve una lista de dicts (una por fold) con RMSE, MAE y R².
    """
    resultados = []

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        resultados.append({
            "fold"   : i + 1,
            "n_train": len(train_idx),
            "n_test" : len(test_idx),
            "RMSE"   : rmse,
            "MAE"    : mae,
            "R2"     : r2,
        })

    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# ============================================================================
def reportar(config, resultados):
    """Imprime los resultados por fold y el resumen agregado."""
    print("=" * 70)
    print("  BASELINE — RANDOM FOREST")
    print("=" * 70)
    print(f"  Target     : {config['target']}")
    print(f"  Features   : {len(config['features'])}  →  {config['features']}")
    print(f"  Validación : {config['validacion']}  ({config['n_folds']} folds)")
    print(f"  Modelo     : RandomForestRegressor (parámetros por defecto)")

    print("\n" + "-" * 70)
    print(f"  {'Fold':<6}{'n_train':<10}{'n_test':<9}{'RMSE':<10}{'MAE':<10}{'R²':<10}")
    print("-" * 70)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_train']:<10}{r['n_test']:<9}"
              f"{r['RMSE']:<10.4f}{r['MAE']:<10.4f}{r['R2']:<10.4f}")

    # Promedios y desviación
    rmse = np.array([r["RMSE"] for r in resultados])
    mae  = np.array([r["MAE"]  for r in resultados])
    r2   = np.array([r["R2"]   for r in resultados])

    print("-" * 70)
    print(f"  {'MEDIA':<25}{rmse.mean():<10.4f}{mae.mean():<10.4f}{r2.mean():<10.4f}")
    print(f"  {'DESV. EST.':<25}{rmse.std():<10.4f}{mae.std():<10.4f}{r2.std():<10.4f}")
    print("=" * 70)



# ============================================================================
# BLOQUE 6 — ORQUESTADOR (main)
# ============================================================================
def main():
    X, y, df = cargar_datos(CONFIG)
    folds    = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 PIPELINE BASELINE — RANDOM FOREST + ESTACIÓN (opcional)
==============================================================================
 Variante del baseline que permite agregar la variable categórica Estacion
 mediante one-hot encoding, activable desde la configuración.

 BLOQUES:
   1 — Configuración        2 — Carga + (one-hot estación)
   3 — Validación           4 — Entrenamiento/evaluación
   5 — Reporte              6 — Main
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Total_Parasitos_",       #Hembras_Ovigeras o "Total_Parasitos_"
    "features"   : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],

    # --- Estación ---
    "usar_estacion" : True,          # True = agrega Estacion con one-hot
    "col_estacion"  : "Estacion",    # nombre de la columna categórica

    # --- Validación ---
    "validacion" : "aleatoria",      # "temporal" o "aleatoria"
    "n_folds"    : 5,
    "random_state": 42,

    "rf_params"  : {
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN (con one-hot de estación)
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    # Solo registros con el target medido
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    # Lista de features que se irá completando
    features = list(config["features"])

    # One-hot de la estación (si está activado)
    if config["usar_estacion"]:
        col = config["col_estacion"]
        dummies = pd.get_dummies(df[col], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)   # agrega es_Verano, es_Otoño, etc.

    X = df[features].copy()
    y = df[config["target"]].copy()

    return X, y, df, features


# ============================================================================
# BLOQUE 3 — VALIDACIÓN
# ============================================================================
def obtener_folds(config, X):
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")
    return list(splitter.split(X))


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    resultados = []
    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        resultados.append({
            "fold"   : i + 1,
            "n_train": len(train_idx),
            "n_test" : len(test_idx),
            "RMSE"   : np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE"    : mean_absolute_error(y_test, y_pred),
            "R2"     : r2_score(y_test, y_pred),
        })
    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE
# ============================================================================
def reportar(config, resultados, features):
    print("=" * 70)
    print("  BASELINE — RANDOM FOREST + ESTACIÓN")
    print("=" * 70)
    print(f"  Target     : {config['target']}")
    print(f"  Estación   : {'Sí (one-hot)' if config['usar_estacion'] else 'No'}")
    print(f"  Features   : {len(features)}  →  {features}")
    print(f"  Validación : {config['validacion']}  ({config['n_folds']} folds)")
    print(f"  Modelo     : RandomForestRegressor (parámetros por defecto)")

    print("\n" + "-" * 70)
    print(f"  {'Fold':<6}{'n_train':<10}{'n_test':<9}{'RMSE':<10}{'MAE':<10}{'R²':<10}")
    print("-" * 70)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_train']:<10}{r['n_test']:<9}"
              f"{r['RMSE']:<10.4f}{r['MAE']:<10.4f}{r['R2']:<10.4f}")

    rmse = np.array([r["RMSE"] for r in resultados])
    mae  = np.array([r["MAE"]  for r in resultados])
    r2   = np.array([r["R2"]   for r in resultados])
    print("-" * 70)
    print(f"  {'MEDIA':<25}{rmse.mean():<10.4f}{mae.mean():<10.4f}{r2.mean():<10.4f}")
    print(f"  {'DESV. EST.':<25}{rmse.std():<10.4f}{mae.std():<10.4f}{r2.std():<10.4f}")
    print("=" * 70)


# ============================================================================
# BLOQUE 6 — MAIN
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    folds = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 PIPELINE BASELINE — RANDOM FOREST (Regresión)
==============================================================================
 Objetivo: establecer un modelo de referencia con hiperparámetros por
 defecto para predecir la prevalencia parasitaria a partir de variables
 físicas oceanográficas.

 Diseño modular por bloques independientes:
   BLOQUE 1 — Configuración (única zona a editar)
   BLOQUE 2 — Carga y preparación de datos
   BLOQUE 3 — Estrategia de validación (temporal / aleatoria)
   BLOQUE 4 — Entrenamiento y evaluación
   BLOQUE 5 — Reporte por consola
   BLOQUE 6 — Orquestador (main)

 Sin optimización de hiperparámetros. Sin guardar archivos.
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN  (lo único que editas para experimentar)
# ============================================================================
CONFIG = {
    # --- Datos ---
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Hembras_Ovigeras",       # o "Total_Parasitos_"
    "features"   : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],

    # --- Validación ---
    # "temporal"  : TimeSeriesSplit (varios folds, respeta el tiempo)
    # "aleatoria" : KFold con shuffle (observaciones independientes)
    # "holdout"   : un solo corte temporal; el último tramo va a test
    "validacion" : "holdout",
    "n_folds"    : 5,            # solo aplica a "temporal" y "aleatoria"
    "test_size"  : 0.20,         # solo aplica a "holdout" (0.20 = último 20%)
    "random_state": 42,

    # --- Hiperparámetros del modelo ---
    # Vacío {} = valores por defecto de scikit-learn (baseline limpio)
    "rf_params"  : {
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN DE DATOS
# ============================================================================
def cargar_datos(config):
    """
    Lee el CSV, filtra registros con target presente (descarta descansos),
    ordena cronológicamente y devuelve X, y y el dataframe de referencia.
    """
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    # Solo registros con el target medido (los únicos que sirven para entrenar)
    df = df[df[config["target"]].notna()].copy()

    # Orden cronológico: imprescindible para validación temporal
    df = df.sort_values("Fecha").reset_index(drop=True)

    X = df[config["features"]].copy()
    y = df[config["target"]].copy()

    return X, y, df


# ============================================================================
# BLOQUE 3 — ESTRATEGIA DE VALIDACIÓN
# ============================================================================
def obtener_folds(config, X):
    """
    Devuelve un iterable de (train_idx, test_idx) según la estrategia elegida.
      - "temporal"  : TimeSeriesSplit (varios folds, respeta el orden del tiempo)
      - "aleatoria" : KFold con shuffle (observaciones independientes)
      - "holdout"   : un solo corte temporal; el último test_size va a prueba.

    Nota: X ya viene ordenado cronológicamente desde cargar_datos(), por lo que
    para el holdout basta con cortar por posición: las primeras filas son las
    más antiguas y las últimas las más recientes.
    """
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
        return list(splitter.split(X))

    elif config["validacion"] == "aleatoria":
        splitter = KFold(
            n_splits=config["n_folds"],
            shuffle=True,
            random_state=config["random_state"],
        )
        return list(splitter.split(X))

    elif config["validacion"] == "holdout":
        n = len(X)
        corte = int(n * (1 - config["test_size"]))   # punto de corte temporal
        train_idx = np.arange(0, corte)              # tramo antiguo → entrenamiento
        test_idx  = np.arange(corte, n)              # tramo reciente → prueba
        return [(train_idx, test_idx)]               # un solo par train/test

    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    """
    Entrena un RandomForestRegressor en cada fold y calcula métricas.
    Devuelve una lista de dicts (una por fold) con RMSE, MAE y R².
    """
    resultados = []

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        resultados.append({
            "fold"   : i + 1,
            "n_train": len(train_idx),
            "n_test" : len(test_idx),
            "RMSE"   : rmse,
            "MAE"    : mae,
            "R2"     : r2,
        })

    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# ============================================================================
def reportar(config, resultados, X=None, df=None, folds=None):
    """Imprime los resultados por fold y el resumen agregado."""
    print("=" * 70)
    print("  BASELINE — RANDOM FOREST")
    print("=" * 70)
    print(f"  Target     : {config['target']}")
    print(f"  Features   : {len(config['features'])}  →  {config['features']}")
    if config["validacion"] == "holdout":
        print(f"  Validación : holdout (último {int(config['test_size']*100)}% temporal)")
        # Mostrar fechas del corte si tenemos el df
        if df is not None and folds is not None:
            train_idx, test_idx = folds[0]
            f_tr = df.iloc[train_idx]["Fecha"]
            f_te = df.iloc[test_idx]["Fecha"]
            print(f"  Train      : {f_tr.min().date()} a {f_tr.max().date()}")
            print(f"  Test       : {f_te.min().date()} a {f_te.max().date()}")
    else:
        print(f"  Validación : {config['validacion']}  ({config['n_folds']} folds)")
    print(f"  Modelo     : RandomForestRegressor (parámetros por defecto)")

    print("\n" + "-" * 70)
    print(f"  {'Fold':<6}{'n_train':<10}{'n_test':<9}{'RMSE':<10}{'MAE':<10}{'R²':<10}")
    print("-" * 70)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_train']:<10}{r['n_test']:<9}"
              f"{r['RMSE']:<10.4f}{r['MAE']:<10.4f}{r['R2']:<10.4f}")

    # Promedios y desviación
    rmse = np.array([r["RMSE"] for r in resultados])
    mae  = np.array([r["MAE"]  for r in resultados])
    r2   = np.array([r["R2"]   for r in resultados])

    print("-" * 70)
    if len(resultados) > 1:
        print(f"  {'MEDIA':<25}{rmse.mean():<10.4f}{mae.mean():<10.4f}{r2.mean():<10.4f}")
        print(f"  {'DESV. EST.':<25}{rmse.std():<10.4f}{mae.std():<10.4f}{r2.std():<10.4f}")
    print("=" * 70)


# ============================================================================
# BLOQUE 6 — ORQUESTADOR (main)
# ============================================================================
def main():
    X, y, df = cargar_datos(CONFIG)
    folds    = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, X=X, df=df, folds=folds)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 PIPELINE BASELINE — RANDOM FOREST (Regresión)
==============================================================================
 Objetivo: establecer un modelo de referencia con hiperparámetros por
 defecto para predecir la prevalencia parasitaria a partir de variables
 físicas oceanográficas.

 Diseño modular por bloques independientes:
   BLOQUE 1 — Configuración (única zona a editar)
   BLOQUE 2 — Carga y preparación de datos
   BLOQUE 3 — Estrategia de validación (temporal / aleatoria)
   BLOQUE 4 — Entrenamiento y evaluación
   BLOQUE 5 — Reporte por consola
   BLOQUE 6 — Orquestador (main)

 Sin optimización de hiperparámetros. Sin guardar archivos.
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN  (lo único que editas para experimentar)
# ============================================================================
CONFIG = {
    # --- Datos ---
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Hembras_Ovigeras",       # o "Total_Parasitos_"
    "log_target" : True,
    "features"   : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],

    # --- Validación ---
    # "temporal"  : TimeSeriesSplit (varios folds, respeta el tiempo)
    # "aleatoria" : KFold con shuffle (observaciones independientes)
    # "holdout"   : un solo corte temporal; el último tramo va a test
    "validacion" : "aleatoria",
    "n_folds"    : 5,            # solo aplica a "temporal" y "aleatoria"
    "test_size"  : 0.20,         # solo aplica a "holdout" (0.20 = último 20%)
    "random_state": 42,

    # --- Hiperparámetros del modelo ---
    # Vacío {} = valores por defecto de scikit-learn (baseline limpio)
    "rf_params"  : {
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN DE DATOS
# ============================================================================
def cargar_datos(config):
    """
    Lee el CSV, filtra registros con target presente (descarta descansos),
    ordena cronológicamente y devuelve X, y y el dataframe de referencia.
    """
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    # Solo registros con el target medido (los únicos que sirven para entrenar)
    df = df[df[config["target"]].notna()].copy()

    # Orden cronológico: imprescindible para validación temporal
    df = df.sort_values("Fecha").reset_index(drop=True)

    X = df[config["features"]].copy()
    y = df[config["target"]].copy()

    return X, y, df


# ============================================================================
# BLOQUE 3 — ESTRATEGIA DE VALIDACIÓN
# ============================================================================
def obtener_folds(config, X):
    """
    Devuelve un iterable de (train_idx, test_idx) según la estrategia elegida.
      - "temporal"  : TimeSeriesSplit (varios folds, respeta el orden del tiempo)
      - "aleatoria" : KFold con shuffle (observaciones independientes)
      - "holdout"   : un solo corte temporal; el último test_size va a prueba.

    Nota: X ya viene ordenado cronológicamente desde cargar_datos(), por lo que
    para el holdout basta con cortar por posición: las primeras filas son las
    más antiguas y las últimas las más recientes.
    """
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
        return list(splitter.split(X))

    elif config["validacion"] == "aleatoria":
        splitter = KFold(
            n_splits=config["n_folds"],
            shuffle=True,
            random_state=config["random_state"],
        )
        return list(splitter.split(X))

    elif config["validacion"] == "holdout":
        n = len(X)
        corte = int(n * (1 - config["test_size"]))   # punto de corte temporal
        train_idx = np.arange(0, corte)              # tramo antiguo → entrenamiento
        test_idx  = np.arange(corte, n)              # tramo reciente → prueba
        return [(train_idx, test_idx)]               # un solo par train/test

    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    """
    Entrena un RandomForestRegressor en cada fold y calcula métricas.
    Si config["log_target"] es True, entrena en escala log(1+x) y revierte
    las predicciones a la escala original antes de medir, para que las
    métricas sigan siendo interpretables en las unidades del target.
    """
    resultados = []

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Transformación del target (solo afecta el entrenamiento)
        if config["log_target"]:
            y_train_fit = np.log1p(y_train)   # log(1 + y)
        else:
            y_train_fit = y_train

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(X_train, y_train_fit)
        y_pred_fit = modelo.predict(X_test)

        # Revertir a escala original si se transformó
        if config["log_target"]:
            y_pred = np.expm1(y_pred_fit)     # inverso de log1p
            y_pred = np.clip(y_pred, 0, None) # evitar negativos por seguridad
        else:
            y_pred = y_pred_fit

        # Métricas SIEMPRE en escala original (interpretables)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        resultados.append({
            "fold"   : i + 1,
            "n_train": len(train_idx),
            "n_test" : len(test_idx),
            "RMSE"   : rmse,
            "MAE"    : mae,
            "R2"     : r2,
        })

    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# ============================================================================
def reportar(config, resultados, X=None, df=None, folds=None):
    """Imprime los resultados por fold y el resumen agregado."""
    print("=" * 70)
    print("  BASELINE — RANDOM FOREST")
    print("=" * 70)
    print(f"  Target     : {config['target']}")
    print(f"  Features   : {len(config['features'])}  →  {config['features']}")
    if config["validacion"] == "holdout":
        print(f"  Validación : holdout (último {int(config['test_size']*100)}% temporal)")
        # Mostrar fechas del corte si tenemos el df
        if df is not None and folds is not None:
            train_idx, test_idx = folds[0]
            f_tr = df.iloc[train_idx]["Fecha"]
            f_te = df.iloc[test_idx]["Fecha"]
            print(f"  Train      : {f_tr.min().date()} a {f_tr.max().date()}")
            print(f"  Test       : {f_te.min().date()} a {f_te.max().date()}")
    else:
        print(f"  Validación : {config['validacion']}  ({config['n_folds']} folds)")
    print(f"  Modelo     : RandomForestRegressor (parámetros por defecto)")

    print("\n" + "-" * 70)
    print(f"  {'Fold':<6}{'n_train':<10}{'n_test':<9}{'RMSE':<10}{'MAE':<10}{'R²':<10}")
    print("-" * 70)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_train']:<10}{r['n_test']:<9}"
              f"{r['RMSE']:<10.4f}{r['MAE']:<10.4f}{r['R2']:<10.4f}")

    # Promedios y desviación
    rmse = np.array([r["RMSE"] for r in resultados])
    mae  = np.array([r["MAE"]  for r in resultados])
    r2   = np.array([r["R2"]   for r in resultados])

    print("-" * 70)
    if len(resultados) > 1:
        print(f"  {'MEDIA':<25}{rmse.mean():<10.4f}{mae.mean():<10.4f}{r2.mean():<10.4f}")
        print(f"  {'DESV. EST.':<25}{rmse.std():<10.4f}{mae.std():<10.4f}{r2.std():<10.4f}")
    print("=" * 70)

    print("\n  Interpretación rápida:")
    print("   - RMSE/MAE: error de predicción en las unidades del target")
    print("     (cuanto más bajo, mejor). RMSE penaliza más los errores grandes.")
    print("   - R²: proporción de varianza explicada (1 = perfecto, 0 = no mejor")
    print("     que predecir la media, negativo = peor que la media).")
    print("   - Alta desviación entre folds = sensibilidad al período evaluado.")


# ============================================================================
# BLOQUE 6 — ORQUESTADOR (main)
# ============================================================================
def main():
    X, y, df = cargar_datos(CONFIG)
    folds    = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, X=X, df=df, folds=folds)


if __name__ == "__main__":
    main()

A
A
A
A
***Lags Temporales***
A
A
A
A

In [ ]:
"""
==============================================================================
 PIPELINE v2 — RANDOM FOREST con REZAGOS, GAP y ESTACIÓN OPCIONAL
==============================================================================
 Versión enriquecida del baseline. Separada del baseline limpio para
 poder comparar honestamente ambas versiones.

 Novedades respecto al baseline:
   - Rezagos (lags) de las variables físicas, calculados POR CENTRO
     respetando la continuidad temporal (los descansos rompen el lag).
   - Validación temporal con GAP (separación entre train y test para
     romper la autocorrelación en la frontera).
   - Estación opcional, codificada de forma CÍCLICA (seno/coseno).

 Diseño modular por bloques:
   BLOQUE 1 — Configuración (única zona a editar)
   BLOQUE 2 — Carga de datos
   BLOQUE 3 — Ingeniería de features (lags + estación cíclica)
   BLOQUE 4 — Estrategia de validación (con gap)
   BLOQUE 5 — Entrenamiento y evaluación
   BLOQUE 6 — Reporte por consola
   BLOQUE 7 — Orquestador (main)

 Sin guardar archivos.
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN  (lo único que editas para experimentar)
# ============================================================================
CONFIG = {
    # --- Datos ---
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Hembras_Ovigeras",       # Hembras_Ovigeras o "Total_Parasitos_"
    "features_base" : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],
    

    # --- Ingeniería de features ---
    "usar_lags"   : True,            # activa/desactiva los rezagos
    "lags_dias"   : [ 7, 14, 28, 56,120],        # 14d=2 sem, 28d=4 sem (agregar 56 para 8 sem)
    "usar_estacion": True,          # estación cíclica (seno/coseno de la semana)

    # --- Validación ---
    # "temporal_gap" : TimeSeriesSplit con separación (gap) entre train y test
    # "aleatoria"    : KFold con shuffle (referencia optimista)
    # "holdout"      : un solo corte temporal final
    "validacion" : "temporal_gap",
    "n_folds"    : 5,
    "gap_dias"   : 14,               # separación train/test (solo temporal_gap)
    "test_size"  : 0.40,             # solo holdout
    "random_state": 42,

    # --- Hiperparámetros del modelo (baseline limpio: solo lo esencial) ---
    "rf_params"  : {
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ============================================================================
# BLOQUE 2 — CARGA DE DATOS
# ============================================================================
def cargar_datos(config):
    """Lee el CSV y devuelve el dataframe completo con Fecha como datetime."""
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    return df


# ============================================================================
# BLOQUE 3 — INGENIERÍA DE FEATURES
# ============================================================================
def construir_lags(df, features_base, lags_dias):
    """
    Crea columnas de rezago para cada variable física, calculadas POR CENTRO
    y respetando la continuidad temporal real.

    Un lag de N días busca el valor de esa variable N días calendario antes,
    en el MISMO centro. Si no existe un registro exactamente N días atrás
    (por ejemplo por un descanso), el lag queda como NaN y ese registro se
    descartará más adelante. Esto evita arrastrar valores de hace meses como
    si fueran "la semana pasada".
    """
    df = df.sort_values(["id_centro", "Fecha"]).reset_index(drop=True)
    nuevas_cols = []

    for centro, grupo in df.groupby("id_centro"):
        # Indexar por fecha para poder buscar desplazamientos calendario exactos
        g = grupo.set_index("Fecha")
        for var in features_base:
            for lag in lags_dias:
                col = f"{var}_lag{lag}d"
                # buscar el valor exactamente 'lag' días antes (misma fecha - lag)
                desplazado = g[var].reindex(g.index - pd.Timedelta(days=lag))
                desplazado.index = g.index   # realinear al índice original
                df.loc[grupo.index, col] = desplazado.values
                if col not in nuevas_cols:
                    nuevas_cols.append(col)

    return df, nuevas_cols


def construir_estacion_ciclica(df):
    """
    Codifica la posición en el año de forma cíclica con seno y coseno,
    de modo que fin y comienzo de año queden próximos en el espacio de
    features. Usa la semana del año (1-53).
    """
    semana = df["Fecha"].dt.isocalendar().week.astype(int)
    df["estacion_sin"] = np.sin(2 * np.pi * semana / 52.0)
    df["estacion_cos"] = np.cos(2 * np.pi * semana / 52.0)
    return df, ["estacion_sin", "estacion_cos"]


def preparar_features(config, df):
    """
    Orquesta la ingeniería de features y devuelve X, y, df_filtrado y la
    lista final de columnas usadas como features.
    Filtra a registros con target presente y sin NaN en las features.
    """
    features = list(config["features_base"])

    # Lags (opcional)
    if config["usar_lags"]:
        df, cols_lag = construir_lags(df, config["features_base"], config["lags_dias"])
        features += cols_lag

    # Estación cíclica (opcional)
    if config["usar_estacion"]:
        df, cols_est = construir_estacion_ciclica(df)
        features += cols_est

    # Quedarnos solo con registros que tienen el target medido
    df = df[df[config["target"]].notna()].copy()

    # Eliminar filas con NaN en alguna feature (por ejemplo lags sin continuidad)
    df = df.dropna(subset=features).copy()

    # Orden cronológico global para la validación temporal
    df = df.sort_values("Fecha").reset_index(drop=True)

    X = df[features].copy()
    y = df[config["target"]].copy()

    return X, y, df, features


# ============================================================================
# BLOQUE 4 — ESTRATEGIA DE VALIDACIÓN
# ============================================================================
def obtener_folds(config, X, df):
    """
    Devuelve lista de (train_idx, test_idx).
      - "temporal_gap": TimeSeriesSplit con gap. El gap se traduce de días a
        número aproximado de muestras usando la densidad media de registros.
      - "aleatoria"   : KFold con shuffle.
      - "holdout"     : un solo corte temporal final.
    """
    if config["validacion"] == "temporal_gap":
        # Estimar cuántas muestras equivalen al gap en días
        dias_totales = (df["Fecha"].max() - df["Fecha"].min()).days
        muestras_por_dia = len(df) / max(dias_totales, 1)
        gap_muestras = int(round(config["gap_dias"] * muestras_por_dia))

        splitter = TimeSeriesSplit(n_splits=config["n_folds"], gap=gap_muestras)
        return list(splitter.split(X)), gap_muestras

    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
        return list(splitter.split(X)), 0

    elif config["validacion"] == "holdout":
        n = len(X)
        corte = int(n * (1 - config["test_size"]))
        return [(np.arange(0, corte), np.arange(corte, n))], 0

    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")


# ============================================================================
# BLOQUE 5 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    """Entrena RF en cada fold y calcula RMSE, MAE y R² en escala original."""
    resultados = []
    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        resultados.append({
            "fold"   : i + 1,
            "n_train": len(train_idx),
            "n_test" : len(test_idx),
            "RMSE"   : np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE"    : mean_absolute_error(y_test, y_pred),
            "R2"     : r2_score(y_test, y_pred),
        })
    return resultados


# ============================================================================
# BLOQUE 6 — REPORTE POR CONSOLA
# ============================================================================
def reportar(config, resultados, features, gap_muestras=0):
    print("=" * 72)
    print("  RANDOM FOREST v2 — Rezagos + Gap + Estación opcional")
    print("=" * 72)
    print(f"  Target        : {config['target']}")
    print(f"  Features base : {len(config['features_base'])}")
    print(f"  Lags          : {'Sí ' + str(config['lags_dias']) + ' días' if config['usar_lags'] else 'No'}")
    print(f"  Estación      : {'Sí (cíclica)' if config['usar_estacion'] else 'No'}")
    print(f"  Total features: {len(features)}")
    print(f"  Validación    : {config['validacion']}")
    if config["validacion"] == "temporal_gap":
        print(f"  Gap           : {config['gap_dias']} días (~{gap_muestras} muestras)")
    print(f"  Modelo        : RandomForestRegressor (parámetros por defecto)")

    print("\n" + "-" * 72)
    print(f"  {'Fold':<6}{'n_train':<10}{'n_test':<9}{'RMSE':<10}{'MAE':<10}{'R²':<10}")
    print("-" * 72)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_train']:<10}{r['n_test']:<9}"
              f"{r['RMSE']:<10.4f}{r['MAE']:<10.4f}{r['R2']:<10.4f}")

    if len(resultados) > 1:
        rmse = np.array([r["RMSE"] for r in resultados])
        mae  = np.array([r["MAE"]  for r in resultados])
        r2   = np.array([r["R2"]   for r in resultados])
        print("-" * 72)
        print(f"  {'MEDIA':<25}{rmse.mean():<10.4f}{mae.mean():<10.4f}{r2.mean():<10.4f}")
        print(f"  {'DESV. EST.':<25}{rmse.std():<10.4f}{mae.std():<10.4f}{r2.std():<10.4f}")
    print("=" * 72)


# ============================================================================
# BLOQUE 7 — ORQUESTADOR (main)
# ============================================================================
def main():
    df = cargar_datos(CONFIG)
    X, y, df_final, features = preparar_features(CONFIG, df)

    print(f"[info] Registros tras ingeniería de features: {len(df_final)}")
    print(f"[info] (algunos se descartan si un lag no tiene continuidad temporal)\n")

    folds, gap_muestras = obtener_folds(CONFIG, X, df_final)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features, gap_muestras)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 TESTS DE VALIDEZ — ¿La validación aleatoria da resultados falsos?
==============================================================================
 Tres pruebas empíricas para comprobar si el R² alto de la validación
 aleatoria es real o un artefacto de fuga de información temporal.

   TEST 1 — Permutación del target:
       Se revuelve el target al azar (se destruye toda relación real).
       Un modelo honesto debe dar R² ~0 o negativo. Si la validación
       aleatoria sigue dando R² alto, está capturando estructura espuria.

   TEST 2 — Aleatoria CON vs SIN lags:
       Cuantifica cuánto del R² de la aleatoria se debe únicamente a
       agregar lags (sospechosos de filtrar información entre train/test).

   TEST 3 — Gap creciente en validación temporal:
       Aumenta la separación train/test. Si el R² temporal permanece
       negativo/estable, confirma que no hay señal temporal aprovechable.

 Sin guardar archivos. Solo consola.
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Hembras_Ovigeras",      # o "Total_Parasitos_"
    "features_base" : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "lags_dias"   : [14, 28],
    "n_folds"     : 5,
    "random_state": 42,
    "rf_params"   : {"random_state": 42, "n_jobs": -1},
}


# ----------------------------------------------------------------------------
# Utilidades de datos
# ----------------------------------------------------------------------------
def cargar(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    return df


def construir_lags(df, features_base, lags_dias):
    df = df.sort_values(["id_centro", "Fecha"]).reset_index(drop=True)
    cols = []
    for centro, grupo in df.groupby("id_centro"):
        g = grupo.set_index("Fecha")
        for var in features_base:
            for lag in lags_dias:
                col = f"{var}_lag{lag}d"
                desplazado = g[var].reindex(g.index - pd.Timedelta(days=lag))
                desplazado.index = g.index
                df.loc[grupo.index, col] = desplazado.values
                if col not in cols:
                    cols.append(col)
    return df, cols


def preparar(config, con_lags):
    df = cargar(config)
    features = list(config["features_base"])
    if con_lags:
        df, cols_lag = construir_lags(df, config["features_base"], config["lags_dias"])
        features += cols_lag
    df = df[df[config["target"]].notna()].copy()
    df = df.dropna(subset=features).copy()
    df = df.sort_values("Fecha").reset_index(drop=True)
    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


def evaluar(X, y, folds, rf_params):
    """Devuelve lista de R² por fold."""
    r2s = []
    for train_idx, test_idx in folds:
        m = RandomForestRegressor(**rf_params)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = m.predict(X.iloc[test_idx])
        r2s.append(r2_score(y.iloc[test_idx], pred))
    return np.array(r2s)


# ============================================================================
# TEST 1 — PERMUTACIÓN DEL TARGET
# ============================================================================
def test_1_permutacion(config):
    print("=" * 72)
    print("  TEST 1 — PERMUTACIÓN DEL TARGET")
    print("=" * 72)
    print("  Se revuelve el target al azar para destruir toda relación real.")
    print("  Un modelo honesto debe dar R² ~0 o negativo.\n")

    X, y, df, feats = preparar(config, con_lags=True)
    rng = np.random.RandomState(config["random_state"])

    # Target REAL vs target PERMUTADO, ambos en validación aleatoria
    kf = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])
    folds = list(kf.split(X))

    r2_real = evaluar(X, y, folds, config["rf_params"])

    y_perm = pd.Series(rng.permutation(y.values), index=y.index)
    r2_perm = evaluar(X, y_perm, folds, config["rf_params"])

    print(f"  {'':<22}{'R² medio':<12}{'R² por fold'}")
    print(f"  {'Target REAL':<22}{r2_real.mean():<12.4f}{np.round(r2_real,3)}")
    print(f"  {'Target PERMUTADO':<22}{r2_perm.mean():<12.4f}{np.round(r2_perm,3)}")

    print("\n  Lectura:")
    if r2_perm.mean() > 0.1:
        print(f"  ⚠ El target permutado AÚN da R²={r2_perm.mean():.3f} > 0.")
        print("    Esto es IMPOSIBLE si no hubiera fuga: sin relación real")
        print("    el modelo no debería predecir nada. CONFIRMA fuga de")
        print("    información en la validación aleatoria.")
    else:
        print(f"  ✓ El target permutado da R²={r2_perm.mean():.3f} ~0.")
        print("    La validación aleatoria NO muestra fuga evidente por esta vía.")
    print()


# ============================================================================
# TEST 2 — ALEATORIA CON VS SIN LAGS
# ============================================================================
def test_2_lags(config):
    print("=" * 72)
    print("  TEST 2 — VALIDACIÓN ALEATORIA: CON vs SIN LAGS")
    print("=" * 72)
    print("  Cuantifica cuánto del R² se debe solo a agregar lags.\n")

    kf = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])

    # Sin lags
    Xs, ys, _, fs = preparar(config, con_lags=False)
    r2_sin = evaluar(Xs, ys, list(kf.split(Xs)), config["rf_params"])

    # Con lags
    Xc, yc, _, fc = preparar(config, con_lags=True)
    r2_con = evaluar(Xc, yc, list(kf.split(Xc)), config["rf_params"])

    print(f"  {'':<22}{'R² medio':<12}{'n features'}")
    print(f"  {'SIN lags':<22}{r2_sin.mean():<12.4f}{len(fs)}")
    print(f"  {'CON lags':<22}{r2_con.mean():<12.4f}{len(fc)}")
    print(f"  {'Salto por lags':<22}{r2_con.mean()-r2_sin.mean():<12.4f}")

    print("\n  Lectura:")
    salto = r2_con.mean() - r2_sin.mean()
    if salto > 0.15:
        print(f"  ⚠ Los lags suben el R² en {salto:.3f} SOLO en validación")
        print("    aleatoria. Como en validación temporal los lags NO ayudan,")
        print("    este salto es fuga: días vecinos comparten información vía lags.")
    else:
        print(f"  El salto por lags es moderado ({salto:.3f}).")
    print()


# ============================================================================
# TEST 3 — GAP CRECIENTE EN VALIDACIÓN TEMPORAL
# ============================================================================
def test_3_gap(config):
    print("=" * 72)
    print("  TEST 3 — GAP CRECIENTE EN VALIDACIÓN TEMPORAL")
    print("=" * 72)
    print("  Si el R² temporal sigue negativo al aumentar el gap, confirma")
    print("  que no hay señal temporal aprovechable real.\n")

    X, y, df, feats = preparar(config, con_lags=True)
    dias_totales = (df["Fecha"].max() - df["Fecha"].min()).days
    muestras_por_dia = len(df) / max(dias_totales, 1)

    print(f"  {'Gap (días)':<14}{'Gap (muestras)':<16}{'R² medio':<12}{'MAE medio'}")
    print("  " + "-" * 54)
    for gap_dias in [0, 14, 28, 56]:
        gap_m = int(round(gap_dias * muestras_por_dia))
        tscv = TimeSeriesSplit(n_splits=config["n_folds"], gap=gap_m)
        folds = list(tscv.split(X))
        r2s, maes = [], []
        for tr, te in folds:
            m = RandomForestRegressor(**config["rf_params"])
            m.fit(X.iloc[tr], y.iloc[tr])
            pred = m.predict(X.iloc[te])
            r2s.append(r2_score(y.iloc[te], pred))
            maes.append(mean_absolute_error(y.iloc[te], pred))
        print(f"  {gap_dias:<14}{gap_m:<16}{np.mean(r2s):<12.4f}{np.mean(maes):.4f}")

    print("\n  Lectura: si el R² permanece negativo o cercano a cero en todos")
    print("  los gaps, el modelo no logra generalizar temporalmente, sin importar")
    print("  la separación. Esto respalda el enfoque interpretativo (SHAP).")
    print()


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("\n" + "#" * 72)
    print(f"#  DIAGNÓSTICO DE VALIDEZ — target: {CONFIG['target']}")
    print("#" * 72 + "\n")
    test_1_permutacion(CONFIG)
    test_2_lags(CONFIG)
    test_3_gap(CONFIG)
    print("=" * 72)
    print("  CONCLUSIÓN GENERAL")
    print("=" * 72)
    print("  - Si Test 1 da R²>0 con target permutado  → hay fuga en aleatoria.")
    print("  - Si Test 2 muestra salto grande por lags → los lags filtran info.")
    print("  - Si Test 3 sigue negativo con todo gap   → no hay predicción temporal.")
    print("  Los tres juntos prueban si el R² alto de la aleatoria es falso.")
    print("=" * 72)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 PIPELINE BASELINE — RANDOM FOREST + ESTACIÓN + MÉTRICAS AMPLIADAS
==============================================================================
 Métricas de regresión:
   RMSE, MAE, R²  (ya existentes)
   MedAE         — mediana del error absoluto (robusta a picos extremos)
   RMSE_norm     — RMSE / media del target (adimensional, comparable
                   entre targets de distinta escala)

 BLOQUES:
   1 — Configuración        2 — Carga + (one-hot estación)
   3 — Validación           4 — Entrenamiento/evaluación
   5 — Reporte              6 — Main
==============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             median_absolute_error, r2_score)
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Total_Parasitos_",       # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"   : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],

    # --- Estación ---
    "usar_estacion" : False,
    "col_estacion"  : "Estacion",

    # --- Validación ---
    "validacion" : "aleatoria",      # "temporal" o "aleatoria"
    "n_folds"    : 5,
    "random_state": 42,

    "rf_params"  : {
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])

    if config["usar_estacion"]:
        col = config["col_estacion"]
        dummies = pd.get_dummies(df[col], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()

    return X, y, df, features


# ============================================================================
# BLOQUE 3 — VALIDACIÓN
# ============================================================================
def obtener_folds(config, X):
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")
    return list(splitter.split(X))


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    resultados = []
    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
        mae   = mean_absolute_error(y_test, y_pred)
        r2    = r2_score(y_test, y_pred)
        medae = median_absolute_error(y_test, y_pred)        # mediana del error absoluto

        # RMSE normalizado: RMSE dividido por la media del target del test.
        # Protección por si la media fuera 0 (evita división por cero).
        media_y = y_test.mean()
        rmse_norm = rmse / media_y if media_y != 0 else np.nan

        resultados.append({
            "fold"     : i + 1,
            "n_train"  : len(train_idx),
            "n_test"   : len(test_idx),
            "RMSE"     : rmse,
            "MAE"      : mae,
            "R2"       : r2,
            "MedAE"    : medae,
            "RMSE_norm": rmse_norm,
        })
    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE
# ============================================================================
def reportar(config, resultados, features):
    print("=" * 86)
    print("  BASELINE — RANDOM FOREST + ESTACIÓN  (métricas ampliadas)")
    print("=" * 86)
    print(f"  Target     : {config['target']}")
    print(f"  Estación   : {'Sí (one-hot)' if config['usar_estacion'] else 'No'}")
    print(f"  Features   : {len(features)}  →  {features}")
    print(f"  Validación : {config['validacion']}  ({config['n_folds']} folds)")
    print(f"  Modelo     : RandomForestRegressor (parámetros por defecto)")

    print("\n" + "-" * 86)
    print(f"  {'Fold':<6}{'n_test':<8}{'RMSE':<10}{'MAE':<10}{'R²':<10}"
          f"{'MedAE':<10}{'RMSE_norm':<12}")
    print("-" * 86)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_test']:<8}{r['RMSE']:<10.4f}{r['MAE']:<10.4f}"
              f"{r['R2']:<10.4f}{r['MedAE']:<10.4f}{r['RMSE_norm']:<12.4f}")

    def col(name): return np.array([r[name] for r in resultados])
    print("-" * 86)
    print(f"  {'MEDIA':<14}{col('RMSE').mean():<10.4f}{col('MAE').mean():<10.4f}"
          f"{col('R2').mean():<10.4f}{col('MedAE').mean():<10.4f}"
          f"{col('RMSE_norm').mean():<12.4f}")
    print(f"  {'DESV. EST.':<14}{col('RMSE').std():<10.4f}{col('MAE').std():<10.4f}"
          f"{col('R2').std():<10.4f}{col('MedAE').std():<10.4f}"
          f"{col('RMSE_norm').std():<12.4f}")
    print("=" * 86)

    print("\n  Guía de lectura:")
    print("   - MedAE < MAE indica que unos pocos picos grandes inflan el MAE;")
    print("     la mayoría de los errores son menores que el MAE promedio.")
    print("   - RMSE_norm expresa el RMSE como fracción de la media del target,")
    print("     permitiendo comparar entre targets de distinta escala.")


# ============================================================================
# BLOQUE 6 — MAIN
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    folds = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 RANDOM FOREST — Baseline con registro de ejecuciones en Excel
==============================================================================
 Cada ejecución se AÑADE a un Excel acumulativo (no lo sobrescribe), para
 llevar una bitácora comparable de todas las configuraciones probadas.

 Hojas del Excel:
   - Ejecuciones    : historial; una fila por corrida (config + métricas)
   - Detalle_Folds  : desglose por fold de la última corrida
   - Importancia    : (preparada) significancia de cada variable
   - SHAP           : (preparada) resultados de SHAP value

 BLOQUES:
   1 Configuración  2 Carga/preparación  3 Validación
   4 Entrenamiento/evaluación  5 Reporte consola  6 Guardado Excel  7 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             median_absolute_error, r2_score)
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Todo lo editable para cada experimento vive aquí.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_RandomForest.xlsx",   # bitácora acumulativa
    "target"       : "Total_Parasitos_",     # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],
    "usar_estacion": True,
    "col_estacion" : "Estacion",
    "validacion"   : "aleatoria",            # "temporal" o "aleatoria"
    "n_folds"      : 5,
    "random_state" : 42,
    "rf_params"    : {
        "random_state": 42,
        "n_jobs": -1,
    },
    "nota"         : "",                      # texto libre para identificar la corrida
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# Filtra registros con target, ordena por fecha y arma la matriz de features
# (con one-hot de estación si está activada).
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — VALIDACIÓN
# Devuelve los pares (train, test) según la estrategia elegida.
# ============================================================================
def obtener_folds(config, X):
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")
    return list(splitter.split(X))


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# Entrena un RF por fold y calcula las cinco métricas de regresión.
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    resultados = []
    for i, (tr, te) in enumerate(folds):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(Xtr, ytr)
        pred = modelo.predict(Xte)

        rmse  = np.sqrt(mean_squared_error(yte, pred))
        media = yte.mean()
        resultados.append({
            "fold": i + 1, "n_train": len(tr), "n_test": len(te),
            "RMSE": rmse,
            "MAE": mean_absolute_error(yte, pred),
            "R2": r2_score(yte, pred),
            "MedAE": median_absolute_error(yte, pred),
            "RMSE_norm": rmse / media if media != 0 else np.nan,
        })
    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# Imprime la tabla por fold y los promedios (lectura rápida en terminal).
# ============================================================================
def reportar(config, resultados, features):
    print("=" * 86)
    print("  RANDOM FOREST —", config["target"],
          "| estación:", "sí" if config["usar_estacion"] else "no",
          "| validación:", config["validacion"])
    print("=" * 86)
    print(f"  {'Fold':<6}{'n_test':<8}{'RMSE':<10}{'MAE':<10}{'R²':<10}"
          f"{'MedAE':<10}{'RMSE_norm':<12}")
    print("-" * 86)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_test']:<8}{r['RMSE']:<10.4f}{r['MAE']:<10.4f}"
              f"{r['R2']:<10.4f}{r['MedAE']:<10.4f}{r['RMSE_norm']:<12.4f}")
    col = lambda k: np.array([r[k] for r in resultados])
    print("-" * 86)
    print(f"  {'MEDIA':<14}{col('RMSE').mean():<10.4f}{col('MAE').mean():<10.4f}"
          f"{col('R2').mean():<10.4f}{col('MedAE').mean():<10.4f}"
          f"{col('RMSE_norm').mean():<12.4f}")
    print("=" * 86)


# ============================================================================
# BLOQUE 6 — GUARDADO EN EXCEL (acumulativo)
# Si el Excel no existe, lo crea con sus hojas. Si existe, AÑADE la corrida
# a "Ejecuciones" y reemplaza "Detalle_Folds" con el desglose más reciente.
# Las hojas "Importancia" y "SHAP" se dejan preparadas para llenarse luego.
# ============================================================================
def guardar_excel(config, resultados, features):
    ruta = config["excel_salida"]
    col = lambda k: np.array([r[k] for r in resultados])

    # Fila resumen de esta ejecución (config + métricas promedio)
    fila = {
        "Fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Modelo": "RandomForest",
        "Target": config["target"],
        "Estacion": "Sí" if config["usar_estacion"] else "No",
        "Validacion": config["validacion"],
        "N_folds": config["n_folds"],
        "N_features": len(features),
        "Hiperparametros": str(config["rf_params"]),
        "RMSE": round(col("RMSE").mean(), 4),
        "MAE": round(col("MAE").mean(), 4),
        "R2": round(col("R2").mean(), 4),
        "MedAE": round(col("MedAE").mean(), 4),
        "RMSE_norm": round(col("RMSE_norm").mean(), 4),
        "R2_desv": round(col("R2").std(), 4),
        "Nota": config["nota"],
    }

    encabezado_fill = PatternFill("solid", start_color="1F4E78")
    encabezado_font = Font(bold=True, color="FFFFFF", name="Arial")

    # Crear el archivo con sus hojas si aún no existe
    if not os.path.exists(ruta):
        wb = Workbook()
        ws = wb.active
        ws.title = "Ejecuciones"
        ws.append(list(fila.keys()))
        for c in ws[1]:
            c.fill = encabezado_fill; c.font = encabezado_font
            c.alignment = Alignment(horizontal="center")

        # Hoja de detalle por fold
        wb.create_sheet("Detalle_Folds")

        # Hoja preparada para importancia de variables
        ws_imp = wb.create_sheet("Importancia")
        ws_imp.append(["Fecha_ejecucion", "Target", "Variable",
                       "Importancia_RF", "Ranking"])
        for c in ws_imp[1]:
            c.fill = encabezado_fill; c.font = encabezado_font

        # Hoja preparada para SHAP
        ws_shap = wb.create_sheet("SHAP")
        ws_shap.append(["Fecha_ejecucion", "Target", "Variable",
                        "SHAP_medio_abs", "Direccion", "Ranking"])
        for c in ws_shap[1]:
            c.fill = encabezado_fill; c.font = encabezado_font

        wb.save(ruta)

    # Abrir el archivo y AÑADIR la nueva ejecución
    wb = load_workbook(ruta)
    wb["Ejecuciones"].append(list(fila.values()))

    # Reemplazar el detalle por fold con el de esta corrida
    ws_det = wb["Detalle_Folds"]
    ws_det.delete_rows(1, ws_det.max_row)   # limpiar lo anterior
    cabecera = ["Fold", "n_train", "n_test", "RMSE", "MAE",
                "R2", "MedAE", "RMSE_norm"]
    ws_det.append(cabecera)
    for c in ws_det[1]:
        c.fill = encabezado_fill; c.font = encabezado_font
    for r in resultados:
        ws_det.append([r["fold"], r["n_train"], r["n_test"],
                       round(r["RMSE"], 4), round(r["MAE"], 4),
                       round(r["R2"], 4), round(r["MedAE"], 4),
                       round(r["RMSE_norm"], 4)])

    # Ajuste simple de ancho de columnas en Ejecuciones
    ws_ej = wb["Ejecuciones"]
    for col_cells in ws_ej.columns:
        ancho = max(len(str(c.value)) for c in col_cells if c.value) + 2
        ws_ej.column_dimensions[col_cells[0].column_letter].width = min(ancho, 40)

    wb.save(ruta)
    print(f"\n[Excel] Ejecución registrada en: {ruta}")


# ============================================================================
# BLOQUE 7 — MAIN
# Orquesta el flujo completo: datos → folds → evaluación → consola → Excel.
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    folds = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features)
    guardar_excel(CONFIG, resultados, features)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 COMPARACIÓN TEMPORAL — Predicción vs Realidad (RF y XGBoost)
==============================================================================
 Genera un gráfico por cada centro de cultivo, comparando los valores
 reales de carga parasitaria contra las predicciones de Random Forest y
 XGBoost superpuestas, a lo largo del tiempo.

 Configuración: Total_Parasitos_, CON estación (la de mejor desempeño).

 IMPORTANTE (honestidad metodológica): las predicciones se obtienen por
 validación cruzada (cross_val_predict), de modo que cada punto es
 predicho por un modelo que NO lo tuvo en su entrenamiento. Esto refleja
 la capacidad real de generalización, no la memorización.

 Requiere: pip install xgboost

 BLOQUES:
   1 Configuración   2 Carga/preparación   3 Predicciones out-of-fold
   4 Gráficos por centro   5 Main
==============================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict, KFold
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Mejor configuración: Total_Parasitos_ con estación, hiperparámetros óptimos.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "carpeta_img"  : "Comparacion_Temporal",
    "target"       : "Total_Parasitos_",
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "col_estacion" : "Estacion",
    "col_region"   : "Region",
    "col_centro"   : "id_centro",
    "col_fecha"    : "Fecha",
    "n_folds"      : 5,
    "random_state" : 42,
    "dias_hueco"      : 21,   # huecos > 21 días se cortan (descansos)
    "ventana_suavizado": 7,   # media móvil de 7 días (semanal)

    # Hiperparámetros óptimos (Total_Parasitos_ CON estación)
    "rf_params"  : {"max_depth": 20, "max_features": 1.0,
                    "min_samples_split": 2, "n_estimators": 300},
    "xgb_params" : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                    "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN (con one-hot de estación)
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df[config["col_fecha"]] = pd.to_datetime(df[config["col_fecha"]])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values(config["col_fecha"]).reset_index(drop=True)

    features = list(config["features"])
    dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
    df = pd.concat([df, dummies], axis=1)
    features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — PREDICCIONES OUT-OF-FOLD (honestas)
# Cada dato es predicho por un modelo que no lo vio en entrenamiento.
# ============================================================================
def predicciones_oof(config, X, y):
    cv = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])

    rf = RandomForestRegressor(random_state=config["random_state"],
                               n_jobs=-1, **config["rf_params"])
    xgb = XGBRegressor(random_state=config["random_state"], n_jobs=-1,
                       objective="reg:squarederror", **config["xgb_params"])

    print("  Generando predicciones out-of-fold (Random Forest)...")
    pred_rf = cross_val_predict(rf, X, y, cv=cv, n_jobs=-1)
    print("  Generando predicciones out-of-fold (XGBoost)...")
    pred_xgb = cross_val_predict(xgb, X, y, cv=cv, n_jobs=-1)

    return pred_rf, pred_xgb


# ============================================================================
# BLOQUE 4 — GRÁFICOS POR CENTRO
# Un gráfico por centro: real vs RF vs XGBoost a lo largo del tiempo.
# Mejoras visuales:
#   - Corte de huecos: se insertan NaN donde hay descansos productivos
#     (saltos temporales grandes), evitando líneas diagonales falsas.
#   - Suavizado: media móvil semanal para reducir el ruido diario y
#     dejar visible la tendencia.
# ============================================================================
def graficar_por_centro(config, df, y, pred_rf, pred_xgb):
    import os
    os.makedirs(config["carpeta_img"], exist_ok=True)

    df = df.copy()
    df["_real"] = y.values
    df["_pred_rf"] = pred_rf
    df["_pred_xgb"] = pred_xgb

    # Umbral de hueco: si entre dos mediciones consecutivas hay más de
    # este número de días, se considera un descanso y se corta la línea.
    dias_hueco = config.get("dias_hueco", 21)
    ventana = config.get("ventana_suavizado", 7)   # días de media móvil

    centros = df[config["col_centro"]].unique()
    for centro in centros:
        sub = df[df[config["col_centro"]] == centro].sort_values(
            config["col_fecha"]).copy()
        region = sub[config["col_region"]].iloc[0]

        # Suavizado por media móvil (centrada) de cada serie
        for col in ["_real", "_pred_rf", "_pred_xgb"]:
            sub[col + "_s"] = sub[col].rolling(window=ventana, center=True,
                                               min_periods=1).mean()

        # Detección de huecos: inserta NaN donde el salto temporal es grande,
        # de modo que matplotlib no dibuje una línea a través del descanso.
        fechas = sub[config["col_fecha"]].values
        saltos = np.diff(fechas).astype("timedelta64[D]").astype(int)
        idx_corte = np.where(saltos > dias_hueco)[0]
        for col in ["_real_s", "_pred_rf_s", "_pred_xgb_s"]:
            vals = sub[col].values.astype(float).copy()
            vals[idx_corte] = np.nan   # rompe la línea justo antes del hueco
            sub[col] = vals

        plt.figure(figsize=(13, 5))
        plt.plot(sub[config["col_fecha"]], sub["_real_s"],
                 color="black", linewidth=1.8, label="Real", alpha=0.85)
        plt.plot(sub[config["col_fecha"]], sub["_pred_rf_s"],
                 color="#1F77B4", linewidth=1.5, label="Random Forest",
                 alpha=0.8)
        plt.plot(sub[config["col_fecha"]], sub["_pred_xgb_s"],
                 color="#D62728", linewidth=1.5, label="XGBoost",
                 alpha=0.8, linestyle="--")

        plt.title(f"Predicción vs Realidad — Centro {centro} ({region})\n"
                  f"Total de parásitos · media móvil semanal · "
                  f"predicciones out-of-fold")
        plt.xlabel("Fecha")
        plt.ylabel("Total de parásitos")
        plt.legend()
        plt.tight_layout()
        salida = f"{config['carpeta_img']}/prediccion_temporal_{centro}.png"
        plt.savefig(salida, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Guardado: {salida}")


# ============================================================================
# BLOQUE 5 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("  COMPARACIÓN TEMPORAL — Predicción vs Realidad (RF y XGBoost)")
    print("=" * 70)
    X, y, df, features = cargar_datos(CONFIG)
    print(f"  Registros: {len(df)} | Features: {len(features)}")
    pred_rf, pred_xgb = predicciones_oof(CONFIG, X, y)
    graficar_por_centro(CONFIG, df, y, pred_rf, pred_xgb)
    print("\n  Completado.")


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 COMPARACIÓN TEMPORAL SIN ESTACIÓN — Predicción vs Realidad (RF y XGBoost)
==============================================================================
 Versión para EXPLORACIÓN: usa SOLO las 5 variables físicas (sin estación)
 y MUESTRA los gráficos en pantalla sin guardarlos en disco.

 Sirve para comparar visualmente si las predicciones sin la estación
 siguen la realidad tan bien como las que la incluían.

 Requiere: pip install xgboost
==============================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict, KFold
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN (sin estación; hiperparámetros óptimos SIN estación)
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"       : "Total_Parasitos_",
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "col_region"   : "Region",
    "col_centro"   : "id_centro",
    "col_fecha"    : "Fecha",
    "n_folds"      : 5,
    "random_state" : 42,
    "dias_hueco"      : 21,
    "ventana_suavizado": 7,

    # Hiperparámetros óptimos de Total_Parasitos_ SIN estación
    "rf_params"  : {"max_depth": None, "max_features": 1.0,
                    "min_samples_split": 2, "n_estimators": 500},
    "xgb_params" : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                    "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
}


# ============================================================================
# BLOQUE 2 — CARGA (sin one-hot de estación: solo físicas)
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df[config["col_fecha"]] = pd.to_datetime(df[config["col_fecha"]])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values(config["col_fecha"]).reset_index(drop=True)

    features = list(config["features"])   # SOLO físicas, sin estación
    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — PREDICCIONES OUT-OF-FOLD
# ============================================================================
def predicciones_oof(config, X, y):
    cv = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])
    rf = RandomForestRegressor(random_state=config["random_state"],
                               n_jobs=-1, **config["rf_params"])
    xgb = XGBRegressor(random_state=config["random_state"], n_jobs=-1,
                       objective="reg:squarederror", **config["xgb_params"])
    print("  Prediciendo out-of-fold (Random Forest, sin estación)...")
    pred_rf = cross_val_predict(rf, X, y, cv=cv, n_jobs=-1)
    print("  Prediciendo out-of-fold (XGBoost, sin estación)...")
    pred_xgb = cross_val_predict(xgb, X, y, cv=cv, n_jobs=-1)
    return pred_rf, pred_xgb


# ============================================================================
# BLOQUE 4 — GRÁFICOS POR CENTRO (se MUESTRAN, no se guardan)
# ============================================================================
def graficar_por_centro(config, df, y, pred_rf, pred_xgb):
    df = df.copy()
    df["_real"] = y.values
    df["_pred_rf"] = pred_rf
    df["_pred_xgb"] = pred_xgb

    dias_hueco = config.get("dias_hueco", 21)
    ventana = config.get("ventana_suavizado", 7)

    centros = df[config["col_centro"]].unique()
    for centro in centros:
        sub = df[df[config["col_centro"]] == centro].sort_values(
            config["col_fecha"]).copy()
        region = sub[config["col_region"]].iloc[0]

        for col in ["_real", "_pred_rf", "_pred_xgb"]:
            sub[col + "_s"] = sub[col].rolling(window=ventana, center=True,
                                               min_periods=1).mean()

        fechas = sub[config["col_fecha"]].values
        saltos = np.diff(fechas).astype("timedelta64[D]").astype(int)
        idx_corte = np.where(saltos > dias_hueco)[0]
        for col in ["_real_s", "_pred_rf_s", "_pred_xgb_s"]:
            vals = sub[col].values.astype(float).copy()
            vals[idx_corte] = np.nan
            sub[col] = vals

        plt.figure(figsize=(13, 5))
        plt.plot(sub[config["col_fecha"]], sub["_real_s"],
                 color="black", linewidth=1.8, label="Real", alpha=0.85)
        plt.plot(sub[config["col_fecha"]], sub["_pred_rf_s"],
                 color="#1F77B4", linewidth=1.5, label="Random Forest",
                 alpha=0.8)
        plt.plot(sub[config["col_fecha"]], sub["_pred_xgb_s"],
                 color="#D62728", linewidth=1.5, label="XGBoost",
                 alpha=0.8, linestyle="--")
        plt.title(f"Predicción vs Realidad (SIN estación) — Centro {centro} ({region})\n"
                  f"Total de parásitos · media móvil semanal · out-of-fold")
        plt.xlabel("Fecha")
        plt.ylabel("Total de parásitos")
        plt.legend()
        plt.tight_layout()
        plt.show()   # MUESTRA en pantalla, NO guarda


# ============================================================================
# BLOQUE 5 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("  COMPARACIÓN TEMPORAL SIN ESTACIÓN (solo físicas)")
    print("=" * 70)
    X, y, df, features = cargar_datos(CONFIG)
    print(f"  Registros: {len(df)} | Features: {len(features)} (solo físicas)")
    pred_rf, pred_xgb = predicciones_oof(CONFIG, X, y)
    graficar_por_centro(CONFIG, df, y, pred_rf, pred_xgb)
    print("\n  Completado (gráficos mostrados, no guardados).")


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 CARGA vs VARIABLE FÍSICA — Real vs Predicho (RF y XGBoost)
==============================================================================
 A partir de las predicciones out-of-fold del total de parásitos, genera
 gráficos de dispersión que muestran cómo se relaciona la carga (real y
 predicha) con cada variable física (temperatura y salinidad), por centro.

 El objetivo es evaluar si el modelo reproduce la relación física-parásito
 esperada: por ejemplo, si captura el umbral térmico o la asociación con
 la salinidad observada en los perfiles de infestación.

 Genera un gráfico por centro y por variable física.

 Requiere: pip install xgboost

 BLOQUES:
   1 Configuración   2 Carga/preparación   3 Predicciones out-of-fold
   4 Scatter carga vs variable   5 Main
==============================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict, KFold
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "carpeta_img"  : "Carga_vs_Fisica",
    "target"       : "Total_Parasitos_",
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    # Variables físicas contra las que se grafica la carga
    "vars_grafico" : ["Temperatura_C", "Salinidad_psu"],
    "col_estacion" : "Estacion",
    "col_region"   : "Region",
    "col_centro"   : "id_centro",
    "col_fecha"    : "Fecha",
    "n_folds"      : 5,
    "random_state" : 42,

    # Hiperparámetros óptimos (Total_Parasitos_ CON estación)
    "rf_params"  : {"max_depth": 20, "max_features": 1.0,
                    "min_samples_split": 2, "n_estimators": 300},
    "xgb_params" : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                    "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN (con one-hot de estación)
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df[config["col_fecha"]] = pd.to_datetime(df[config["col_fecha"]])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values(config["col_fecha"]).reset_index(drop=True)

    features = list(config["features"])
    dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
    df = pd.concat([df, dummies], axis=1)
    features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — PREDICCIONES OUT-OF-FOLD
# ============================================================================
def predicciones_oof(config, X, y):
    cv = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])
    rf = RandomForestRegressor(random_state=config["random_state"],
                               n_jobs=-1, **config["rf_params"])
    xgb = XGBRegressor(random_state=config["random_state"], n_jobs=-1,
                       objective="reg:squarederror", **config["xgb_params"])
    print("  Prediciendo out-of-fold (Random Forest)...")
    pred_rf = cross_val_predict(rf, X, y, cv=cv, n_jobs=-1)
    print("  Prediciendo out-of-fold (XGBoost)...")
    pred_xgb = cross_val_predict(xgb, X, y, cv=cv, n_jobs=-1)
    return pred_rf, pred_xgb


# ============================================================================
# BLOQUE 4 — SCATTER: CARGA vs VARIABLE FÍSICA (por centro y variable)
# ============================================================================
def graficar_carga_vs_fisica(config, df, y, pred_rf, pred_xgb):
    import os
    os.makedirs(config["carpeta_img"], exist_ok=True)

    df = df.copy()
    df["_real"] = y.values
    df["_pred_rf"] = pred_rf
    df["_pred_xgb"] = pred_xgb

    # Etiquetas legibles para los ejes
    etiquetas = {
        "Temperatura_C": "Temperatura (°C)",
        "Salinidad_psu": "Salinidad (psu)",
    }

    centros = df[config["col_centro"]].unique()
    for centro in centros:
        sub = df[df[config["col_centro"]] == centro]
        region = sub[config["col_region"]].iloc[0]

        for var in config["vars_grafico"]:
            plt.figure(figsize=(9, 6))

            # Puntos reales (referencia) y predichos por cada modelo
            plt.scatter(sub[var], sub["_real"], s=18, alpha=0.45,
                        color="black", label="Real")
            plt.scatter(sub[var], sub["_pred_rf"], s=18, alpha=0.45,
                        color="#1F77B4", label="Random Forest", marker="^")
            plt.scatter(sub[var], sub["_pred_xgb"], s=18, alpha=0.45,
                        color="#D62728", label="XGBoost", marker="s")

            plt.title(f"Carga parasitaria vs {etiquetas.get(var, var)}\n"
                      f"Centro {centro} ({region}) · real vs predicho "
                      f"(out-of-fold)")
            plt.xlabel(etiquetas.get(var, var))
            plt.ylabel("Total de parásitos")
            plt.legend()
            plt.tight_layout()

            nombre_var = var.replace("_", "")
            salida = f"{config['carpeta_img']}/carga_vs_{nombre_var}_{centro}.png"
            plt.savefig(salida, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"  Guardado: {salida}")


# ============================================================================
# BLOQUE 5 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("  CARGA vs VARIABLE FÍSICA — Real vs Predicho")
    print("=" * 70)
    X, y, df, features = cargar_datos(CONFIG)
    print(f"  Registros: {len(df)} | Features: {len(features)}")
    pred_rf, pred_xgb = predicciones_oof(CONFIG, X, y)
    graficar_carga_vs_fisica(CONFIG, df, y, pred_rf, pred_xgb)
    print("\n  Completado.")


if __name__ == "__main__":
    main()